## Phase 5 - Model Improvement(GRU)

In [1]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Dense,
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    GlobalAveragePooling1D,
    Dropout
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
data = pd.read_csv("imdb_cleaned.csv")
print(data.shape)

(49582, 4)


In [3]:
X = data["clean_review"]
y = data["label"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.50,
    random_state=42,
    stratify=y_test
)

print(len(X_train))
print(len(X_val))
print(len(X_test))

39665
4958
4959


In [5]:
import pickle

with open("tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

print(len(tokenizer.word_index))

90662


In [6]:
MAX_SEQUENCE_LENGTH = 500

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_val_sequences = tokenizer.texts_to_sequences(X_val)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

X_train_integer = pad_sequences(
    X_train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_integer = pad_sequences(
    X_val_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_integer = pad_sequences(
    X_test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

y_train = np.asarray(y_train)
y_val = np.asarray(y_val)
y_test = np.asarray(y_test)

print(X_train_integer.shape)
print(X_val_integer.shape)
print(X_test_integer.shape)

(39665, 500)
(4958, 500)
(4959, 500)


In [7]:
def evaluate_model(model, X_test, y_test, model_name, batch_size=64):

    # Generate probabilities
    probabilities = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0
    ).ravel()

    # Convert probabilities to class predictions
    predictions = (probabilities >= 0.5).astype(int)

    # metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test,  predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_test, probabilities)

    # Confusion Matrix
    cm = confusion_matrix(y_test, predictions)

    # Classification Report
    report = classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"],
        digits=4
    )

    # Print results
    print("=" * 60)
    print(f"{model_name} RESULTS")
    print("=" * 60)

    print(f"Accuracy : {accuracy:.5f}")
    print(f"Precision: {precision:.5f}")
    print(f"Recall   : {recall:.5f}")
    print(f"F1 Score : {f1:.5f}")
    print(f"ROC-AUC  : {roc_auc:.5f}")

    print("\nClassification Report")
    print("-" * 60)
    print(report)

    print("Confusion Matrix")
    print(cm)

    # Return everything for later comparison
    results = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }

    return results, probabilities, predictions, cm

**GRU(EarlyStopping)**

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

gru_es = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_es.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [25]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)
gru_es_history = gru_es.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 52ms/step - accuracy: 0.5046 - loss: 0.6931 - val_accuracy: 0.5040 - val_loss: 0.6922
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5148 - loss: 0.6924 - val_accuracy: 0.5073 - val_loss: 0.6930
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5273 - loss: 0.6699 - val_accuracy: 0.5141 - val_loss: 0.6979
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 51ms/step - accuracy: 0.6194 - loss: 0.6109 - val_accuracy: 0.8292 - val_loss: 0.4450
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.8775 - loss: 0.3090 - val_accuracy: 0.8905 - val_loss: 0.2644
Epoch 6/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9457 - loss: 0.1557 - val_accuracy: 0.9006 - val_loss: 0.2718
Epoch 7/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9735 - loss: 0.0839 - val_accuracy: 0.8911 - val_loss: 0.3504
Epoch 8/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.9859 - loss: 0.0480 - 

In [26]:
gru_early_results, gru_early_probabilities, gru_early_predictions, gru_early_cm = evaluate_model(
    gru_es,
    X_test_integer,
    y_test,
    "GRU EarlyStopping",
    batch_size=64
)

GRU EarlyStopping RESULTS
Accuracy : 0.89272
Precision: 0.86855
Recall   : 0.92648
F1 Score : 0.89658
ROC-AUC  : 0.96061

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9206    0.8587    0.8886      2470
    Positive     0.8685    0.9265    0.8966      2489

    accuracy                         0.8927      4959
   macro avg     0.8946    0.8926    0.8926      4959
weighted avg     0.8945    0.8927    0.8926      4959

Confusion Matrix
[[2121  349]
 [ 183 2306]]


**GRU(Embedding dimention)(256-128)**

In [ ]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 256

gru_dim = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    GRU(128),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

gru_dim.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_dim.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
gru_dim.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_14 (Embedding)        │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_14 (GRU)                    │ (None, 128)            │       148,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,836,545 (29.89 MB)

 Trainable params: 7,836,545 (29.89 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import time

start_time = time.time()

gru_dim_history = gru_dim.fit(
    X_train_integer,
    y_train,
    validation_data=(
        X_val_integer,
        y_val
    ),
    epochs=10,
    batch_size=64,
    verbose=1
)

training_time = time.time() - start_time

print(f"GRU embedding dimention training time: " f"{training_time:.2f} seconds")

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 39ms/step - accuracy: 0.4965 - loss: 0.6933 - val_accuracy: 0.5018 - val_loss: 0.6923
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 37ms/step - accuracy: 0.5158 - loss: 0.6832 - val_accuracy: 0.5083 - val_loss: 0.6951
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.5322 - loss: 0.6559 - val_accuracy: 0.5107 - val_loss: 0.7113
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.7801 - loss: 0.4055 - val_accuracy: 0.8935 - val_loss: 0.2627
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.9471 - loss: 0.1471 - val_accuracy: 0.8838 - val_loss: 0.2946
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 35ms/step - accuracy: 0.9832 - loss: 0.0556 - val_accuracy: 0.8877 - val_loss: 0.3946
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.9945 - loss: 0.0220 - val_accuracy: 0.8903 - val_loss: 0.4329
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9967 - loss: 0.0133 - 

In [ ]:
gru_dim_results, gru_dim_probabilities, gru_dim_predictions, gru_dim_cm = evaluate_model(
    model=gru_dim,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU DIM 128",
    batch_size=64
)

GRU DIM 128 RESULTS
Accuracy : 0.87860
Precision: 0.86330
Recall   : 0.90076
F1 Score : 0.88164
ROC-AUC  : 0.93992

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8954    0.8563    0.8754      2470
    Positive     0.8633    0.9008    0.8816      2489

    accuracy                         0.8786      4959
   macro avg     0.8794    0.8785    0.8785      4959
weighted avg     0.8793    0.8786    0.8785      4959

Confusion Matrix
[[2115  355]
 [ 247 2242]]


**GRU(Embedding dimention)(256-256)**

In [ ]:
VOCAB_SIZE = 30000
EMBEDDING_DIM = 256

gru_dim256 = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),

    GRU(256),

    Dense(64, activation="relu"),

    Dense(1, activation="sigmoid")
])

gru_dim256.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_dim256.build(input_shape=(None, MAX_SEQUENCE_LENGTH))

gru_dim256.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_dim256_history = gru_dim256.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 42s 59ms/step - accuracy: 0.5006 - loss: 0.6933 - val_accuracy: 0.5054 - val_loss: 0.6927
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 58ms/step - accuracy: 0.5150 - loss: 0.6873 - val_accuracy: 0.5036 - val_loss: 0.6955
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.5319 - loss: 0.6631 - val_accuracy: 0.5054 - val_loss: 0.7173
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.5368 - loss: 0.6445 - val_accuracy: 0.5113 - val_loss: 0.7450
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8522 - loss: 0.3070 - val_accuracy: 0.8947 - val_loss: 0.2637
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.9592 - loss: 0.1178 - val_accuracy: 0.8937 - val_loss: 0.3133
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.9873 - loss: 0.0426 - val_accuracy: 0.8832 - val_loss: 0.4160
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.9958 - loss: 0.0161 - 

In [ ]:
gru_dim256_results, gru_dim256_probabilities, gru_dim256_predictions, gru_dim256_cm = evaluate_model(
    model=gru_dim256,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Improved 2",
    batch_size=64
)

GRU Improved 2 RESULTS
Accuracy : 0.88123
Precision: 0.88061
Recall   : 0.88309
F1 Score : 0.88185
ROC-AUC  : 0.94832

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8819    0.8794    0.8806      2470
    Positive     0.8806    0.8831    0.8818      2489

    accuracy                         0.8812      4959
   macro avg     0.8812    0.8812    0.8812      4959
weighted avg     0.8812    0.8812    0.8812      4959

Confusion Matrix
[[2172  298]
 [ 291 2198]]


**GRU(Dropout)**

In [ ]:
gru_dropout = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_dropout.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_dropout.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
gru_dropout.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_dropout_history = gru_dropout.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 53ms/step - accuracy: 0.5028 - loss: 0.6937 - val_accuracy: 0.5020 - val_loss: 0.6925
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.5129 - loss: 0.6881 - val_accuracy: 0.5056 - val_loss: 0.6927
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.5365 - loss: 0.6669 - val_accuracy: 0.7293 - val_loss: 0.5739
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.8737 - loss: 0.3113 - val_accuracy: 0.8985 - val_loss: 0.2562
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.9580 - loss: 0.1218 - val_accuracy: 0.8961 - val_loss: 0.2894
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.9858 - loss: 0.0483 - val_accuracy: 0.8856 - val_loss: 0.3967
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.9943 - loss: 0.0211 - val_accuracy: 0.8893 - val_loss: 0.5140
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.9971 - loss: 0.0112 - 

In [ ]:
gru_dropout_results, gru_dropout_probabilities, gru_dropout_predictions, gru_dropout_cm = evaluate_model(
    model=gru_dropout,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Dropout",
    batch_size=64
)

GRU Dropout RESULTS
Accuracy : 0.86973
Precision: 0.91678
Recall   : 0.81438
F1 Score : 0.86255
ROC-AUC  : 0.94480

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8319    0.9255    0.8762      2470
    Positive     0.9168    0.8144    0.8626      2489

    accuracy                         0.8697      4959
   macro avg     0.8743    0.8699    0.8694      4959
weighted avg     0.8745    0.8697    0.8693      4959

Confusion Matrix
[[2286  184]
 [ 462 2027]]


**GRU(Recurrent Dropout)**

In [55]:
gru_recurrent_dropout = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),

    GRU(
        256,
        dropout=0.3,
        recurrent_dropout=0.2
    ),

    Dense(64, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

gru_recurrent_dropout.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_recurrent_dropout.build(
    input_shape=(None, MAX_SEQUENCE_LENGTH)
)

gru_recurrent_dropout.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_13 (GRU)                    │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [56]:
gru_recurrent_dropout_history = gru_recurrent_dropout.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
202/620 ━━━━━━━━━━━━━━━━━━━━ 10:57 2s/step - accuracy: 0.5089 - loss: 0.6942

KeyboardInterrupt: 

In [ ]:
gru_recurrent_dropout_results, gru_recurrent_dropout_probabilities, gru_recurrent_dropout_predictions, gru_recurrent_dropout_cm = evaluate_model(
    model=gru_recurrent_dropout,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Recurrent Dropout",
    batch_size=64
)

**GRU(RMSPROP)**

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [44]:
gru_rmsprop = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_rmsprop.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_rmsprop.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
gru_rmsprop.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_10 (GRU)                    │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [45]:
gru_rmsprop_history = gru_rmsprop.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 47ms/step - accuracy: 0.5023 - loss: 0.6934 - val_accuracy: 0.5018 - val_loss: 0.6931
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 30s 48ms/step - accuracy: 0.5050 - loss: 0.6928 - val_accuracy: 0.5056 - val_loss: 0.6927
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.5036 - loss: 0.6919 - val_accuracy: 0.5028 - val_loss: 0.6926
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 43s 57ms/step - accuracy: 0.5108 - loss: 0.6897 - val_accuracy: 0.5046 - val_loss: 0.6932
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.5124 - loss: 0.6864 - val_accuracy: 0.5069 - val_loss: 0.6974
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.5177 - loss: 0.6821 - val_accuracy: 0.5101 - val_loss: 0.6937
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5239 - loss: 0.6757 - val_accuracy: 0.5075 - val_loss: 0.6974
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5276 - loss: 0.6668 - 

In [46]:
gru_rmsprop_results, gru_rmsprop_probabilities, gru_rmsprop_predictions, gru_rmsprop_cm = evaluate_model(
    model=gru_rmsprop,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU RMSprop",
    batch_size=64
)

GRU RMSprop RESULTS
Accuracy : 0.51059
Precision: 0.63136
Recall   : 0.05986
F1 Score : 0.10936
ROC-AUC  : 0.51494

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5046    0.9648    0.6626      2470
    Positive     0.6314    0.0599    0.1094      2489

    accuracy                         0.5106      4959
   macro avg     0.5680    0.5123    0.3860      4959
weighted avg     0.5682    0.5106    0.3849      4959

Confusion Matrix
[[2383   87]
 [2340  149]]


**GRU(Batch normalization)**

In [47]:
gru_batchnorm = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_batchnorm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [48]:
gru_batchnorm_history = gru_batchnorm.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 37s 55ms/step - accuracy: 0.4952 - loss: 0.7783 - val_accuracy: 0.4986 - val_loss: 0.6969
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.5068 - loss: 0.7222 - val_accuracy: 0.5103 - val_loss: 0.8940
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.5201 - loss: 0.6936 - val_accuracy: 0.5018 - val_loss: 0.7100
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8212 - loss: 0.3705 - val_accuracy: 0.8741 - val_loss: 0.2903
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.9339 - loss: 0.1737 - val_accuracy: 0.7406 - val_loss: 0.6214
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.9597 - loss: 0.1125 - val_accuracy: 0.8933 - val_loss: 0.2813
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.9757 - loss: 0.0705 - val_accuracy: 0.7013 - val_loss: 1.4114
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.9852 - loss: 0.0440 - 

In [49]:
gru_batchnorm_results, gru_batchnorm_probabilities, gru_batchnorm_predictions, gru_batchnorm_cm = evaluate_model(
    model=gru_batchnorm,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Batch Normalization",
    batch_size=64
)

GRU Batch Normalization RESULTS
Accuracy : 0.79774
Precision: 0.95921
Recall   : 0.62354
F1 Score : 0.75578
ROC-AUC  : 0.95215

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.7195    0.9733    0.8274      2470
    Positive     0.9592    0.6235    0.7558      2489

    accuracy                         0.7977      4959
   macro avg     0.8394    0.7984    0.7916      4959
weighted avg     0.8398    0.7977    0.7915      4959

Confusion Matrix
[[2404   66]
 [ 937 1552]]


**GRU(Learning Rate Tuning)**

In [ ]:
gru_lr = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_lr.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_lr.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
gru_lr.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_7 (GRU)                     │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_lr_history = gru_lr.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 58ms/step - accuracy: 0.9348 - loss: 0.1793 - val_accuracy: 0.8983 - val_loss: 0.2738
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 57ms/step - accuracy: 0.9685 - loss: 0.0967 - val_accuracy: 0.8810 - val_loss: 0.3579
Epoch 3/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 57ms/step - accuracy: 0.9852 - loss: 0.0501 - val_accuracy: 0.8909 - val_loss: 0.4191
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 36s 58ms/step - accuracy: 0.9926 - loss: 0.0267 - val_accuracy: 0.8836 - val_loss: 0.5107
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
gru_lr_results, gru_lr_probabilities, gru_lr_predictions, gru_lr_cm = evaluate_model(
    model=gru_lr,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Learning Rate 0.0005",
    batch_size=64
)

GRU Learning Rate 0.0005 RESULTS
Accuracy : 0.89917
Precision: 0.91180
Recall   : 0.88469
F1 Score : 0.89804
ROC-AUC  : 0.96178

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8872    0.9138    0.9003      2470
    Positive     0.9118    0.8847    0.8980      2489

    accuracy                         0.8992      4959
   macro avg     0.8995    0.8992    0.8992      4959
weighted avg     0.8995    0.8992    0.8992      4959

Confusion Matrix
[[2257  213]
 [ 287 2202]]


**GRU(Learning Rate Scheduling)**

In [39]:
gru_optimized = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3,),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_optimized.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_optimized.build(input_shape=(None, MAX_SEQUENCE_LENGTH))
gru_optimized.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 500, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_9 (GRU)                     │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
    verbose=1
)

In [41]:
gru_optimized_history = gru_optimized.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[reduce_lr],
    verbose=1
)

Epoch 1/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 35s 53ms/step - accuracy: 0.5019 - loss: 0.6933 - val_accuracy: 0.5028 - val_loss: 0.6922 - learning_rate: 5.0000e-04
Epoch 2/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5156 - loss: 0.6905
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5137 - loss: 0.6902 - val_accuracy: 0.5079 - val_loss: 0.6922 - learning_rate: 5.0000e-04
Epoch 3/15
619/620 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5290 - loss: 0.6688
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
620/620 ━━━━━━━━━━━━━━━━━━━━ 31s 50ms/step - accuracy: 0.5286 - loss: 0.6673 - val_accuracy: 0.5097 - val_loss: 0.7048 - learning_rate: 2.5000e-04
Epoch 4/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.5536 - loss: 0.6450 - val_accuracy: 0.7654 - val_loss: 0.6023 - learning_rate: 1.2500e-04
Epoch 5/15
620/620 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy

In [43]:
gru_optimized_results, gru_optimized_probabilities, gru_optimized_predictions, gru_optimized_cm = evaluate_model(
    model=gru_optimized,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU ReduceLR",
    batch_size=64
)

GRU ReduceLR RESULTS
Accuracy : 0.88909
Precision: 0.88858
Recall   : 0.89072
F1 Score : 0.88965
ROC-AUC  : 0.95452

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8896    0.8874    0.8885      2470
    Positive     0.8886    0.8907    0.8896      2489

    accuracy                         0.8891      4959
   macro avg     0.8891    0.8891    0.8891      4959
weighted avg     0.8891    0.8891    0.8891      4959

Confusion Matrix
[[2192  278]
 [ 272 2217]]


**GRU(BATCH SIZE 32)**

In [27]:
gru_batch32 = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_batch32.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [28]:
gru_batch32_history = gru_batch32.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 49s 38ms/step - accuracy: 0.5003 - loss: 0.6936 - val_accuracy: 0.5040 - val_loss: 0.6926
Epoch 2/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 44s 36ms/step - accuracy: 0.5180 - loss: 0.6861 - val_accuracy: 0.5050 - val_loss: 0.6987
Epoch 3/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 45s 36ms/step - accuracy: 0.5271 - loss: 0.6635 - val_accuracy: 0.5099 - val_loss: 0.7039
Epoch 4/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 45s 36ms/step - accuracy: 0.5360 - loss: 0.6458 - val_accuracy: 0.5111 - val_loss: 0.7150
Epoch 5/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 44s 35ms/step - accuracy: 0.5370 - loss: 0.6399 - val_accuracy: 0.5137 - val_loss: 0.7453
Epoch 6/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 44s 36ms/step - accuracy: 0.5407 - loss: 0.6381 - val_accuracy: 0.5153 - val_loss: 0.7689
Epoch 7/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 44s 35ms/step - accuracy: 0.5396 - loss: 0.6374 - val_accuracy: 0.5149 - val_loss: 0.7825
Epoch 8/10
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 83s 37ms/step - accuracy: 0.6412 -

In [29]:
gru_batch32_results, gru_batch32_probabilities, gru_batch32_predictions, gru_batch32_cm = evaluate_model(
    model=gru_batch32,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Batch Size 32",
    batch_size=32
)

GRU Batch Size 32 RESULTS
Accuracy : 0.88869
Precision: 0.85180
Recall   : 0.94215
F1 Score : 0.89470
ROC-AUC  : 0.95401

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.9347    0.8348    0.8820      2470
    Positive     0.8518    0.9421    0.8947      2489

    accuracy                         0.8887      4959
   macro avg     0.8933    0.8885    0.8883      4959
weighted avg     0.8931    0.8887    0.8883      4959

Confusion Matrix
[[2062  408]
 [ 144 2345]]


**GRU(BATCH SIZE 128)**

In [30]:
gru_batch128 = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_batch128.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [31]:
gru_batch128_history = gru_batch128.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=128,
    verbose=1
)

Epoch 1/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 30s 90ms/step - accuracy: 0.5028 - loss: 0.6934 - val_accuracy: 0.5020 - val_loss: 0.6927
Epoch 2/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 27s 85ms/step - accuracy: 0.5124 - loss: 0.6899 - val_accuracy: 0.5089 - val_loss: 0.6915
Epoch 3/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - accuracy: 0.5228 - loss: 0.6787 - val_accuracy: 0.5083 - val_loss: 0.6963
Epoch 4/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 27s 88ms/step - accuracy: 0.5319 - loss: 0.6584 - val_accuracy: 0.5111 - val_loss: 0.7106
Epoch 5/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 28s 89ms/step - accuracy: 0.5372 - loss: 0.6445 - val_accuracy: 0.5123 - val_loss: 0.7411
Epoch 6/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 28s 91ms/step - accuracy: 0.6125 - loss: 0.6038 - val_accuracy: 0.8013 - val_loss: 0.4926
Epoch 7/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 28s 91ms/step - accuracy: 0.8806 - loss: 0.3091 - val_accuracy: 0.8903 - val_loss: 0.2713
Epoch 8/10
310/310 ━━━━━━━━━━━━━━━━━━━━ 28s 90ms/step - accuracy: 0.9433 - loss: 0.1640 - 

In [32]:
gru_batch128_results, gru_batch128_probabilities, gru_batch128_predictions, gru_batch128_cm = evaluate_model(
    model=gru_batch128,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU Batch Size 128",
    batch_size=128
)

GRU Batch Size 128 RESULTS
Accuracy : 0.88465
Precision: 0.89788
Recall   : 0.86902
F1 Score : 0.88322
ROC-AUC  : 0.94920

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8722    0.9004    0.8861      2470
    Positive     0.8979    0.8690    0.8832      2489

    accuracy                         0.8847      4959
   macro avg     0.8850    0.8847    0.8846      4959
weighted avg     0.8851    0.8847    0.8846      4959

Confusion Matrix
[[2224  246]
 [ 326 2163]]


**GRU(SGD)**

In [36]:
gru_sgd = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_sgd.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [37]:
gru_sgd_history = gru_sgd.fit(
    X_train_integer,
    y_train,
    validation_data=(X_val_integer, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 29s 43ms/step - accuracy: 0.5018 - loss: 0.6932 - val_accuracy: 0.5050 - val_loss: 0.6930
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.5019 - loss: 0.6932 - val_accuracy: 0.5052 - val_loss: 0.6930
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.5008 - loss: 0.6932 - val_accuracy: 0.5018 - val_loss: 0.6930
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 46ms/step - accuracy: 0.5025 - loss: 0.6932 - val_accuracy: 0.5052 - val_loss: 0.6931
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.5013 - loss: 0.6931 - val_accuracy: 0.5052 - val_loss: 0.6930
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.5003 - loss: 0.6931 - val_accuracy: 0.5018 - val_loss: 0.6930
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.5033 - loss: 0.6931 - val_accuracy: 0.5050 - val_loss: 0.6929
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 27s 44ms/step - accuracy: 0.5028 - loss: 0.6931 - 

In [38]:
gru_sgd_results, gru_sgd_probabilities, gru_sgd_predictions, gru_sgd_cm = evaluate_model(
    model=gru_sgd,
    X_test=X_test_integer,
    y_test=y_test,
    model_name="GRU SGD",
    batch_size=64
)

GRU SGD RESULTS
Accuracy : 0.50837
Precision: 0.55916
Recall   : 0.09683
F1 Score : 0.16507
ROC-AUC  : 0.50966

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.5035    0.9231    0.6516      2470
    Positive     0.5592    0.0968    0.1651      2489

    accuracy                         0.5084      4959
   macro avg     0.5313    0.5100    0.4083      4959
weighted avg     0.5315    0.5084    0.4074      4959

Confusion Matrix
[[2280  190]
 [2248  241]]


**GRU(Sequensial length 300)**

In [50]:
GRU_SEQUENCE_LENGTH = 300

X_train_gru_300 = pad_sequences(
    X_train_sequences,
    maxlen=GRU_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_val_gru_300 = pad_sequences(
    X_val_sequences,
    maxlen=GRU_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

X_test_gru_300 = pad_sequences(
    X_test_sequences,
    maxlen=GRU_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("X_train:", X_train_gru_300.shape)
print("X_val:", X_val_gru_300.shape)
print("X_test:", X_test_gru_300.shape)

X_train: (39665, 300)
X_val: (4958, 300)
X_test: (4959, 300)


In [51]:
gru_seq300 = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=256
    ),
    GRU(256, dropout=0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_seq300.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gru_seq300.build(input_shape=(None, GRU_SEQUENCE_LENGTH))
gru_seq300.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (None, 300, 256)       │     7,680,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_12 (GRU)                    │ (None, 256)            │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,091,265 (30.87 MB)

 Trainable params: 8,091,265 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

In [52]:
gru_seq300_history = gru_seq300.fit(
    X_train_gru_300,
    y_train,
    validation_data=(X_val_gru_300, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 26s 35ms/step - accuracy: 0.5081 - loss: 0.6934 - val_accuracy: 0.4984 - val_loss: 0.6929
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 40s 33ms/step - accuracy: 0.5425 - loss: 0.6737 - val_accuracy: 0.5010 - val_loss: 0.6985
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.7321 - loss: 0.4701 - val_accuracy: 0.8862 - val_loss: 0.2794
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 42s 34ms/step - accuracy: 0.9271 - loss: 0.1946 - val_accuracy: 0.8965 - val_loss: 0.2772
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9627 - loss: 0.1091 - val_accuracy: 0.8923 - val_loss: 0.2983
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9821 - loss: 0.0592 - val_accuracy: 0.8881 - val_loss: 0.3937
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9905 - loss: 0.0328 - val_accuracy: 0.8778 - val_loss: 0.6179
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.9939 - loss: 0.0227 - 

In [53]:
gru_seq300_results, gru_seq300_probabilities, gru_seq300_predictions, gru_seq300_cm = evaluate_model(
    model=gru_seq300,
    X_test=X_test_gru_300,
    y_test=y_test,
    model_name="GRU Sequence Length 300",
    batch_size=64
)

GRU Sequence Length 300 RESULTS
Accuracy : 0.87820
Precision: 0.85987
Recall   : 0.90478
F1 Score : 0.88175
ROC-AUC  : 0.94636

Classification Report
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative     0.8987    0.8514    0.8744      2470
    Positive     0.8599    0.9048    0.8818      2489

    accuracy                         0.8782      4959
   macro avg     0.8793    0.8781    0.8781      4959
weighted avg     0.8792    0.8782    0.8781      4959

Confusion Matrix
[[2103  367]
 [ 237 2252]]


In [57]:
import pandas as pd

# ============================================
# GRU MODEL COMPARISON RESULTS
# ============================================

gru_results = [
    {
        "Rank": 1,
        "Model": "GRU Baseline",
        "Accuracy": 0.88465,
        "Precision": 0.88202,
        "Recall": 0.88911,
        "F1 Score": 0.88555,
        "ROC-AUC": 0.94349
    },
    {
        "Rank": 2,
        "Model": "GRU EarlyStopping",
        "Accuracy": 0.89272,
        "Precision": 0.86855,
        "Recall": 0.92648,
        "F1 Score": 0.89658,
        "ROC-AUC": 0.96061
    },
    {
        "Rank": 3,
        "Model": "GRU Dim 128",
        "Accuracy": 0.87860,
        "Precision": 0.86330,
        "Recall": 0.90076,
        "F1 Score": 0.88164,
        "ROC-AUC": 0.93992
    },
    {
        "Rank": 4,
        "Model": "GRU Dim 256",
        "Accuracy": 0.88123,
        "Precision": 0.88061,
        "Recall": 0.88309,
        "F1 Score": 0.88185,
        "ROC-AUC": 0.94832
    },
    {
        "Rank": 5,
        "Model": "GRU Dropout",
        "Accuracy": 0.86973,
        "Precision": 0.91678,
        "Recall": 0.81438,
        "F1 Score": 0.86255,
        "ROC-AUC": 0.94480
    },
    {
        "Rank": 6,
        "Model": "GRU RMSprop",
        "Accuracy": 0.51059,
        "Precision": 0.63136,
        "Recall": 0.05986,
        "F1 Score": 0.10936,
        "ROC-AUC": 0.51494
    },
    {
        "Rank": 7,
        "Model": "GRU Batch Normalization",
        "Accuracy": 0.79774,
        "Precision": 0.95921,
        "Recall": 0.62354,
        "F1 Score": 0.75578,
        "ROC-AUC": 0.95215
    },
    {
        "Rank": 8,
        "Model": "GRU Learning Rate 0.0005",
        "Accuracy": 0.89917,
        "Precision": 0.91180,
        "Recall": 0.88469,
        "F1 Score": 0.89804,
        "ROC-AUC": 0.96178
    },
    {
        "Rank": 9,
        "Model": "GRU ReduceLR",
        "Accuracy": 0.88909,
        "Precision": 0.88858,
        "Recall": 0.89072,
        "F1 Score": 0.88965,
        "ROC-AUC": 0.95452
    },
    {
        "Rank": 10,
        "Model": "GRU Batch Size 32",
        "Accuracy": 0.88869,
        "Precision": 0.85180,
        "Recall": 0.94215,
        "F1 Score": 0.89470,
        "ROC-AUC": 0.95401
    },
    {
        "Rank": 11,
        "Model": "GRU Batch Size 128",
        "Accuracy": 0.88465,
        "Precision": 0.89788,
        "Recall": 0.86902,
        "F1 Score": 0.88322,
        "ROC-AUC": 0.94920
    },
    {
        "Rank": 12,
        "Model": "GRU SGD",
        "Accuracy": 0.50837,
        "Precision": 0.55916,
        "Recall": 0.09683,
        "F1 Score": 0.16507,
        "ROC-AUC": 0.50966
    },
    {
        "Rank": 13,
        "Model": "GRU Sequence Length 300",
        "Accuracy": 0.87820,
        "Precision": 0.85987,
        "Recall": 0.90478,
        "F1 Score": 0.88175,
        "ROC-AUC": 0.94636
    }
]

# Create DataFrame
gru_df = pd.DataFrame(gru_results)

# Display the complete table
print("=" * 90)
print("GRU MODEL COMPARISON")
print("=" * 90)

display(gru_df)



GRU MODEL COMPARISON


,Rank,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,1,GRU Baseline,0.88465,0.88202,0.88911,0.88555,0.94349
1,2,GRU EarlyStopping,0.89272,0.86855,0.92648,0.89658,0.96061
2,3,GRU Dim 128,0.87860,0.86330,0.90076,0.88164,0.93992
3,4,GRU Dim 256,0.88123,0.88061,0.88309,0.88185,0.94832
4,5,GRU Dropout,0.86973,0.91678,0.81438,0.86255,0.94480
5,6,GRU RMSprop,0.51059,0.63136,0.05986,0.10936,0.51494
6,7,GRU Batch Normalization,0.79774,0.95921,0.62354,0.75578,0.95215
7,8,GRU Learning Rate 0.0005,0.89917,0.91180,0.88469,0.89804,0.96178
8,9,GRU ReduceLR,0.88909,0.88858,0.89072,0.88965,0.95452
9,10,GRU Batch Size 32,0.88869,0.85180,0.94215,0.89470,0.95401


In [59]:
gru_df.to_csv("gru_model_comparison.csv", index=False)
print("saved ")

saved 
